# Moving Average Strategy

In [1]:
from datetime import datetime
from pathlib import Path
import plotly.graph_objects as go
import pandas as pd
import math
import os

from trading_lab.data import get_stock_data
from trading_lab.backtest import run_backtest

### Stocklist

In [2]:
STOCKS = [
    "APA",
    "CHRD",
]

In [3]:
df = get_stock_data(STOCKS, datetime(2021, 1, 1), datetime(2022, 12, 31))
print(df.shape)
print("Index levels of df:", df.index.nlevels)
print("Columns of df:", df.columns.tolist())

df.head(10)

[DataLoader] Cache hit: a9944d9922_2021-01-01_2022-12-31_all.parquet
(1006, 7)
Index levels of df: 2
Columns of df: ['open', 'high', 'low', 'close', 'volume', 'trade_count', 'vwap']


open   high    low  close      volume  \
symbol timestamp                                                           
APA    2021-01-04 05:00:00+00:00  12.47  12.76  12.27  12.60  10472620.0   
       2021-01-05 05:00:00+00:00  12.68  14.27  12.68  13.81  15616899.0   
       2021-01-06 05:00:00+00:00  14.00  14.57  13.39  14.37  14704273.0   
       2021-01-07 05:00:00+00:00  14.51  14.93  14.28  14.60   9230476.0   
       2021-01-08 05:00:00+00:00  14.81  14.89  13.91  14.15   8138755.0   
       2021-01-11 05:00:00+00:00  13.61  14.29  13.47  14.22   6815200.0   
       2021-01-12 05:00:00+00:00  14.57  15.83  14.43  15.81  12308874.0   
       2021-01-13 05:00:00+00:00  15.79  15.81  15.10  15.19   7698060.0   
       2021-01-14 05:00:00+00:00  15.68  16.49  15.58  15.66  11984266.0   
       2021-01-15 05:00:00+00:00  15.39  15.39  14.61  14.74   9566440.0   

                                  trade_count   vwap  
symbol timestamp                                      
APA    2021-01-04 05:00:00+00:00      54950.0  12.55  
       2021-01-05 05:00:00+00:00      78029.0  13.78  
       2021-01-06 05:00:00+00:00      75823.0  14.21  
       2021-01-07 05:00:00+00:00      48634.0  14.63  
       2021-01-08 05:00:00+00:00      44362.0  14.29  
       2021-01-11 05:00:00+00:00      36343.0  14.01  
       2021-01-12 05:00:00+00:00      68175.0  15.41  
       2021-01-13 05:00:00+00:00      45922.0  15.35  
       2021-01-14 05:00:00+00:00      65008.0  15.90  
       2021-01-15 05:00:00+00:00      56320.0  14.94

### Calulate the moving average of stockprice

The moving average is calcualted as followed:
$$
MA_t=\frac{1}{t}\sum_{i=1}^{t}P_i
$$
, for t-days.

In [4]:
def rolling_ma(df, stocks, days):
    close_prices = (
        df.loc[stocks, "close"]  # Select the 'close' prices for the specified stocks
        .unstack(level="symbol") # Unstack the 'symbol' level to create a DataFrame with symbols as columns
        .sort_index()
    )

    return close_prices.rolling(window=days).mean()

In [5]:
ma_df = rolling_ma(df, STOCKS, 20)
ma_df = ma_df.dropna()
print(ma_df.info())
ma_df.head(10)

<class 'pandas.DataFrame'>
DatetimeIndex: 484 entries, 2021-02-01 05:00:00+00:00 to 2022-12-30 05:00:00+00:00
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   APA     484 non-null    float64
 1   CHRD    484 non-null    float64
dtypes: float64(2)
memory usage: 11.3 KB
None


symbol,APA,CHRD
timestamp,,
2021-02-01 05:00:00+00:00,14.0260,25.0215
2021-02-02 05:00:00+00:00,14.0620,25.0665
2021-02-03 05:00:00+00:00,14.0745,25.1405
2021-02-04 05:00:00+00:00,14.0545,25.1810
2021-02-05 05:00:00+00:00,14.0185,25.2810
2021-02-08 05:00:00+00:00,14.0745,25.4530
2021-02-09 05:00:00+00:00,14.1005,25.6250
2021-02-10 05:00:00+00:00,14.0635,25.7900
2021-02-11 05:00:00+00:00,14.0305,25.9515


In [6]:
close_df = (
    df.loc[STOCKS, "close"]
    .unstack(level="symbol")
    .sort_index()
)

cropped_prices = close_df.loc[ma_df.index]

fig = go.Figure()
for stock in STOCKS:
    stock_bars = df.loc[stock].sort_index()
    fig.add_trace(go.Scatter(
        x=cropped_prices.index,
        y=cropped_prices[stock],
        name=f"{stock} Close Price",
    ))

    fig.add_trace(go.Scatter(
        x=ma_df.index,
        y=ma_df[stock],
        name=f"{stock} 20-Day MA",
    ))
    fig.update_layout(
        title="Close Price and 20-Day Moving Average",
        xaxis_title="Date",
        yaxis_title="Price",
        legend_title="Legend",
        template="plotly_dark"
    )

fig.show()

### Turning the moving average into a trading signal

What is the hypothesis behind this strategy?
This is a trend-following rule. Its hypothesis is:
Price movements sometimes persist long enough that following their direction can be profitable after trading costs.

The moving average smooths recent prices and provides a reference:
- Price above its recent average suggests relative strength.
- Price below its recent average suggests relative weakness.

It does not establish that a stock is cheap or expensive, nor guarantee its next move. Moving averages react to historical prices and therefore lag changes.
The moving average strategy is a simple trend-following rule. It generates a trading signal \(s_t\) based on the relationship between today’s closing price \(C_t\) and its simple moving average over the last n days\(SMA_{n,t}\):
$$
s_t =
\begin{cases}
1 & \text{if } C_t > SMA_{n,t}\\
0 & \text{otherwise}
\end{cases}
$$
Here:
- \(C_t\) is today’s closing price.
- \(SMA_{n,t}\) averages today’s close and the preceding \(n-1\) closes.
- 1 means “I want to hold the stock.”
- 0 means “I want to hold cash.”

Based on this rule, a trade is made when the signal changes:

| Previous target | New target | Action |
|---:|---:|---|
| 0 | 1 | Buy |
| 1 | 1 | Keep holding |
| 1 | 0 | Sell the existing holding |
| 0 | 0 | Stay in cash |

When the signal changes one the closing price the resulting trade is executed at the next day’s opening price. The strategy is applied to each stock in the portfolio independently.

Note:
In sideways markets, prices can repeatedly cross the MA, causing repeated entries and exits. These reversals are often called whipsaws; trading costs make them more expensive. A shorter MA generally reacts faster, while a longer MA smooths more and reacts later. Neither is automatically better.

In [7]:
def ma_signal(stocks, df, days, ma_wide):
    """timestamp | close | ma_20 | target_position"""
    close_wide = df.loc[stocks, "close"].unstack(level="symbol").sort_index()
    target_pos_wide = (close_wide > ma_wide).astype(int)

    close_wide = close_wide.add_suffix("_close")
    ma_wide = ma_wide.add_suffix(f"_ma_{days}")
    target_pos_wide = target_pos_wide.add_suffix("_target_position")

    signal_df = pd.concat([close_wide, ma_wide, target_pos_wide], axis=1).sort_index(axis=1) # axis=1 means concatenate along columns
    return signal_df

ma_wide = rolling_ma(df=df, stocks=STOCKS, days= 20)
ma_signal(STOCKS, df, 20, ma_wide).head(22)

symbol,APA_close,APA_ma_20,APA_target_position,CHRD_close,CHRD_ma_20,CHRD_target_position
timestamp,,,,,,
2021-01-04 05:00:00+00:00,12.60,NaN,0,23.85,NaN,0
2021-01-05 05:00:00+00:00,13.81,NaN,0,24.10,NaN,0
2021-01-06 05:00:00+00:00,14.37,NaN,0,25.55,NaN,0
2021-01-07 05:00:00+00:00,14.60,NaN,0,25.59,NaN,0
2021-01-08 05:00:00+00:00,14.15,NaN,0,25.83,NaN,0
2021-01-11 05:00:00+00:00,14.22,NaN,0,25.92,NaN,0
2021-01-12 05:00:00+00:00,15.81,NaN,0,27.41,NaN,0
2021-01-13 05:00:00+00:00,15.19,NaN,0,27.14,NaN,0
2021-01-14 05:00:00+00:00,15.66,NaN,0,27.83,NaN,0


### Simulation



In [8]:
signals = ma_signal(stocks=STOCKS, df=df, days=20, ma_wide= ma_wide)

position_columns = [
    f"{stock}_target_position"
    for stock in STOCKS
]

target_positions = signals[position_columns].copy()

target_positions = target_positions.rename(
    columns={
        f"{stock}_target_position": stock
        for stock in STOCKS
    }
)
result = run_backtest(
    start_capital=10_000,
    stocks=STOCKS,
    df=df,
    target_positions=target_positions,
)

print(result[0])

{'stocks': {'APA': {'cash': 0.0, 'shares': 183.55964690368978}, 'CHRD': {'cash': 13242.626873724994, 'shares': 0.0}}, 'depo_value': 20744.71}


In [9]:
daily_portfolio_value = result[1]
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=daily_portfolio_value.index,
    y=daily_portfolio_value,
    name="Daily Portfolio Value",
))
fig.update_layout(
    title="Daily Portfolio Value Over Time",
    xaxis_title="Date",
    yaxis_title="Portfolio Value",
    legend_title="Legend",
    template="plotly_dark"
)
fig.show()